In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import euclidean_distances, cosine_similarity

# ── USER CONFIG ──────────────────────────────────────────────────────────
XLSX_FILE = "Konzum_BiH_2015_2024_1H.xlsx"   # workbook with all years
EXPECTED_HOURS = 8760                        # after cleaning every year
LOAD_COL = "KONZUM (MWh)"                   # column with demand values
# ─────────────────────────────────────────────────────────────────────────

def clean_year(sheet_name: str, xls: pd.ExcelFile):
    """Return a 1-D numpy array of 8760 hourly loads, or None if bad."""
    df = pd.read_excel(xls, sheet_name=sheet_name)

    # ── 1. Build a proper timestamp column ──────────────────────────────
    # Convert the 'time' field (often datetime.time objects) to "HH:MM:SS"
    df["time_str"] = df["time"].apply(lambda t: t.strftime("%H:%M:%S")
                                                if not isinstance(t, str) else t)

    # Handle the “24:00:00” convention → set to next-day 00:00
    mask_24 = df["time_str"] == "24:00:00"
    df.loc[mask_24, "time_str"] = "00:00:00"
    df["date_str"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
    df.loc[mask_24, "date_str"] = (
        pd.to_datetime(df.loc[mask_24, "date_str"]) + pd.Timedelta(days=1)
    ).dt.strftime("%Y-%m-%d")

    # Combine and parse
    df["timestamp"] = pd.to_datetime(
        df["date_str"] + " " + df["time_str"],
        format="%Y-%m-%d %H:%M:%S",
        errors="coerce",
    )
    df = df.dropna(subset=["timestamp"])

    # ── 2. Remove 29 Feb so every year has 8760 rows ────────────────────
    df = df[~((df["timestamp"].dt.month == 2) & (df["timestamp"].dt.day == 29))]

    # ── 3. Clean the load column ────────────────────────────────────────
    df[LOAD_COL] = (
        df[LOAD_COL]
        .replace(["n/a", "n/e", "N/A", "N/E", "", " "], np.nan)
        .astype("float64")
    )

    # Interpolate (time-aware) any gaps, then ffill/bfill as last resort
    df = df.set_index("timestamp").sort_index()
    df[LOAD_COL] = (
        df[LOAD_COL]
        .interpolate("time", limit_direction="both")
        .fillna(method="ffill")
        .fillna(method="bfill")
    )

    # ── 4. Sanity check row count ───────────────────────────────────────
    if len(df) != EXPECTED_HOURS:
        print(
            f"⚠️  Sheet '{sheet_name}' skipped "
            f"(rows = {len(df)}, expected {EXPECTED_HOURS})."
        )
        return None

    return df[LOAD_COL].to_numpy()

# ── MAIN PIPELINE ────────────────────────────────────────────────────────
xls = pd.ExcelFile(XLSX_FILE)
profiles = {}  # year → 8760-hour array

for sheet in xls.sheet_names:
    series = clean_year(sheet, xls)
    if series is not None:
        profiles[int(sheet)] = series

if len(profiles) < 2:
    raise RuntimeError("Not enough valid years to compare!")

# Make DataFrame : rows = years, columns = hour-of-year (0…8759)
profiles_df = pd.DataFrame.from_dict(profiles, orient="index")

print("✅ Cleaned data shape (years × hours):", profiles_df.shape)

# ── Normalise each year by its annual total (shape-focused) ─────────────
norm_df = profiles_df.div(profiles_df.sum(axis=1), axis=0)

# ── Representative-year selection ───────────────────────────────────────
eucl_dist = euclidean_distances(norm_df)
cos_dissim = 1 - cosine_similarity(norm_df)

eucl_medoid_year = norm_df.index[np.argmin(eucl_dist.mean(axis=1))]
cos_medoid_year  = norm_df.index[np.argmin(cos_dissim.mean(axis=1))]

print(f"📊 Representative year (Euclidean): {eucl_medoid_year}")
print(f"📊 Representative year (Cosine):    {cos_medoid_year}")

# ── Export medoid profiles for external use ─────────────────────────────
norm_df.loc[eucl_medoid_year].to_csv("euclidean_representative_profile.csv",
                                     index=False)
norm_df.loc[cos_medoid_year].to_csv("cosine_representative_profile.csv",
                                    index=False)
print("✅ Representative profiles saved.")


FileNotFoundError: [Errno 2] No such file or directory: 'Konzum_BiH_2015_2024_1H.xlsx'